In [1]:
import cv2
import numpy as np
import pandas as pd
from openpose import pyopenpose as op

ModuleNotFoundError: No module named 'openpose'

In [2]:
import cv2
import mediapipe as mp
import numpy as np

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

In [3]:
VIDEO_FOLDER = "/Users/jeremiah/video-p-reaction-joketype-modular/fifteen_twenty_five"

In [4]:
def extract_mediapipe_pose(frames):
    # Initialize pose detection
    pose = mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5, min_tracking_confidence=0.5)

    people_features = []
    for frame in frames:
        # Convert frame to RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb_frame)

        # Check if any landmarks are detected
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            # Extract features for each person (for now, just consider the first person)
            person_features = []
            for lm in landmarks:
                person_features.append([lm.x, lm.y, lm.z])  # x, y, z coordinates
            people_features.append(np.array(person_features).flatten())  # Flatten features

    return np.array(people_features)  # Return features for each frame/person

In [5]:
def sample_frames(video_path, n_frames):
    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Adjust if video has fewer frames than requested
    n_frames_to_sample = min(n_frames, frame_count)
    
    for i in range(n_frames_to_sample):
        # Set the position of the next frame to be captured
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_count * i / n_frames_to_sample))
        ret, frame = cap.read()
        if ret:
            # Resize frame to a consistent size
            frame = cv2.resize(frame, (224, 224))
            frames.append(frame)
    
    cap.release()
    
    # If the video has fewer frames than `n_frames`, pad the output with the last valid frame
    if len(frames) < n_frames:
        last_frame = frames[-1] if frames else np.zeros((224, 224, 3), dtype=np.uint8)  # Fallback to blank frame
        frames.extend([last_frame] * (n_frames - len(frames)))
    
    return np.array(frames)


In [23]:
vid_1 = "/Users/jeremiah/video-p-reaction-joketype-modular/fifteen_twenty_five/2UWdXP.joke1.rep2.take1.Peekaboo.mp4"
vid_2 = "/Users/jeremiah/video-p-reaction-joketype-modular/fifteen_twenty_five/7G4D4S.joke4.rep3.take1.NomNomNom.mp4"

In [24]:
import cv2
import pandas as pd
import numpy as np
import torch
import mediapipe as mp
from ultralytics import YOLO

In [17]:
import cv2
import pandas as pd
import numpy as np
import mediapipe as mp
from ultralytics import YOLO

# Initialize MediaPipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, 
                    min_detection_confidence=0.5, 
                    min_tracking_confidence=0.5)

# Define the landmarks we want to track
LANDMARKS = [
    mp_pose.PoseLandmark.NOSE,
    mp_pose.PoseLandmark.LEFT_EYE_INNER,
    mp_pose.PoseLandmark.LEFT_EYE,
    mp_pose.PoseLandmark.LEFT_EYE_OUTER,
    mp_pose.PoseLandmark.RIGHT_EYE_INNER,
    mp_pose.PoseLandmark.RIGHT_EYE,
    mp_pose.PoseLandmark.RIGHT_EYE_OUTER,
    mp_pose.PoseLandmark.LEFT_EAR,
    mp_pose.PoseLandmark.RIGHT_EAR,
    mp_pose.PoseLandmark.MOUTH_LEFT,
    mp_pose.PoseLandmark.MOUTH_RIGHT,
    mp_pose.PoseLandmark.LEFT_SHOULDER,
    mp_pose.PoseLandmark.RIGHT_SHOULDER,
    mp_pose.PoseLandmark.LEFT_ELBOW,
    mp_pose.PoseLandmark.RIGHT_ELBOW,
    mp_pose.PoseLandmark.LEFT_WRIST,
    mp_pose.PoseLandmark.RIGHT_WRIST,
    mp_pose.PoseLandmark.LEFT_PINKY,
    mp_pose.PoseLandmark.RIGHT_PINKY,
    mp_pose.PoseLandmark.LEFT_INDEX,
    mp_pose.PoseLandmark.RIGHT_INDEX,
    mp_pose.PoseLandmark.LEFT_THUMB,
    mp_pose.PoseLandmark.RIGHT_THUMB,
    mp_pose.PoseLandmark.LEFT_HIP,
    mp_pose.PoseLandmark.RIGHT_HIP,
    mp_pose.PoseLandmark.LEFT_KNEE,
    mp_pose.PoseLandmark.RIGHT_KNEE,
    mp_pose.PoseLandmark.LEFT_ANKLE,
    mp_pose.PoseLandmark.RIGHT_ANKLE,
    mp_pose.PoseLandmark.LEFT_HEEL,
    mp_pose.PoseLandmark.RIGHT_HEEL,
    mp_pose.PoseLandmark.LEFT_FOOT_INDEX,
    mp_pose.PoseLandmark.RIGHT_FOOT_INDEX
]

import os
# Load the YOLO model for person detection
yolo_model = YOLO("yolo11n-pose.pt")  # Load your YOLO model

# Function to process the video and extract pose landmarks
def process_video2(video_path):
    video_name,_ = os.path.splitext(video_path)
    cap = cv2.VideoCapture(video_path)
    frame_id = 0
    all_features = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("End of video or unable to read the frame.")
            break

        # Detect persons in the frame using YOLO
        results = yolo_model(frame)

        # Store the coordinates of detected persons and their landmarks
        detected_persons = []

        # Loop through each detection
        for detection in results[0].boxes:
            class_id = int(detection.cls[0])  # Get the class ID
            if class_id == 0:  # Check if the detected object is a person
                # Get bounding box coordinates
                xyxy = detection.xyxy[0].tolist()  # Convert to list
                x1, y1, x2, y2 = map(int, xyxy)  # Ensure we convert to integers

                # Crop the detected person from the frame
                person_frame = frame[y1:y2, x1:x2]

                # Convert the cropped frame to RGB for MediaPipe
                rgb_person_frame = cv2.cvtColor(person_frame, cv2.COLOR_BGR2RGB)

                # Detect pose landmarks using MediaPipe
                pose_results = pose.process(rgb_person_frame)

                if pose_results.pose_landmarks:  # Check if any landmarks were detected
                    keypoints_data = []
                    
                    # Collect keypoint data
                    for landmark in LANDMARKS:
                        landmark_data = pose_results.pose_landmarks.landmark[landmark]
                        keypoints_data.append((landmark_data.x, landmark_data.y, landmark_data.z, landmark_data.visibility))

                    detected_persons.append((x1, y1, x2, y2, keypoints_data))

        # Check if we detected persons and extract their keypoints
        if detected_persons:
            for i, (x1, y1, x2, y2, keypoints) in enumerate(detected_persons):
                row_data = [frame_id, i]  # Frame ID and Person ID (0, 1)
                
                # Assuming keypoints are in the format: [x, y, z, visibility]
                for kp in keypoints:
                    row_data.extend(kp)  # Add x, y, z, visibility to row data

                all_features.append(row_data)
        else:
            print(f"No detected persons in frame {frame_id}.")

        frame_id += 1

    cap.release()
    pose.close()  # Close the MediaPipe Pose object

    # Create DataFrame if we have features collected
    if all_features:
        num_keypoints = len(LANDMARKS)  # Number of landmarks defined in LANDMARKS

        # Create column names using the landmark names
        columns = ['frame_id', 'person'] + \
                  [f'{landmark.name}_x' for landmark in LANDMARKS] + \
                  [f'{landmark.name}_y' for landmark in LANDMARKS] + \
                  [f'{landmark.name}_z' for landmark in LANDMARKS] + \
                  [f'{landmark.name}_visibility' for landmark in LANDMARKS]
        
        df = pd.DataFrame(all_features, columns=columns)

        # Save DataFrame to CSV if not empty
        df.to_csv('mediapipe_pose_landmarks_two_people_per_frame_with_visibility.csv', index=False)
        print("Pose landmarks with visibility saved to mediapipe_pose_landmarks_two_people_per_frame_with_visibility.csv")
    else:
        print("No data collected. The DataFrame is empty.")

I0000 00:00:1730237534.142054 14927701 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3 Pro


W0000 00:00:1730237534.212603 14943740 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1730237534.222328 14943740 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [18]:
vid_3 = "/Users/jeremiah/video-p-reaction-joketype-modular/fifteen_twenty_five/"
import os


In [19]:
video_list = os.listdir(vid_3)

for vid in video_list:
    vid_path = vid_3 + vid
    process_video2(vid_path)


0: 480x640 2 persons, 60.1ms
Speed: 1.2ms preprocess, 60.1ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)



0: 480x640 2 persons, 41.8ms
Speed: 0.9ms preprocess, 41.8ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 persons, 38.7ms
Speed: 1.1ms preprocess, 38.7ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)



/opt/anaconda3/envs/videop/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


0: 480x640 2 persons, 40.2ms
Speed: 0.7ms preprocess, 40.2ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 42.9ms
Speed: 0.7ms preprocess, 42.9ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 49.2ms
Speed: 0.7ms preprocess, 49.2ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 50.7ms
Speed: 0.9ms preprocess, 50.7ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 40.3ms
Speed: 0.9ms preprocess, 40.3ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 44.7ms
Speed: 0.9ms preprocess, 44.7ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 47.3ms
Speed: 1.4ms preprocess, 47.3ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 44.3ms
Speed: 0.6ms preprocess, 44.3ms inference, 0.6ms postprocess per image at shape (1

ValueError: _graph is None in SolutionBase

In [12]:
# Call the function with your video path
process_video2(vid_2)


0: 480x640 2 persons, 57.3ms
Speed: 2.1ms preprocess, 57.3ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 47.4ms
Speed: 0.8ms preprocess, 47.4ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 45.4ms
Speed: 0.9ms preprocess, 45.4ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 42.8ms
Speed: 0.8ms preprocess, 42.8ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 43.5ms
Speed: 0.7ms preprocess, 43.5ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 42.3ms
Speed: 0.8ms preprocess, 42.3ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 37.7ms
Speed: 1.0ms preprocess, 37.7ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 51.5ms
Speed: 0.9ms preprocess, 51.5ms inference, 4.4ms postprocess per image at shape (

In [25]:
import os
import cv2
import pandas as pd
import numpy as np
import mediapipe as mp
from ultralytics import YOLO

# Initialize MediaPipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, 
                    min_detection_confidence=0.5, 
                    min_tracking_confidence=0.5)

# Define the landmarks we want to track
LANDMARKS = [
    mp_pose.PoseLandmark.NOSE,
    mp_pose.PoseLandmark.LEFT_EYE_INNER,
    mp_pose.PoseLandmark.LEFT_EYE,
    mp_pose.PoseLandmark.LEFT_EYE_OUTER,
    mp_pose.PoseLandmark.RIGHT_EYE_INNER,
    mp_pose.PoseLandmark.RIGHT_EYE,
    mp_pose.PoseLandmark.RIGHT_EYE_OUTER,
    mp_pose.PoseLandmark.LEFT_EAR,
    mp_pose.PoseLandmark.RIGHT_EAR,
    mp_pose.PoseLandmark.MOUTH_LEFT,
    mp_pose.PoseLandmark.MOUTH_RIGHT,
    mp_pose.PoseLandmark.LEFT_SHOULDER,
    mp_pose.PoseLandmark.RIGHT_SHOULDER,
    mp_pose.PoseLandmark.LEFT_ELBOW,
    mp_pose.PoseLandmark.RIGHT_ELBOW,
    mp_pose.PoseLandmark.LEFT_WRIST,
    mp_pose.PoseLandmark.RIGHT_WRIST,
    mp_pose.PoseLandmark.LEFT_PINKY,
    mp_pose.PoseLandmark.RIGHT_PINKY,
    mp_pose.PoseLandmark.LEFT_INDEX,
    mp_pose.PoseLandmark.RIGHT_INDEX,
    mp_pose.PoseLandmark.LEFT_THUMB,
    mp_pose.PoseLandmark.RIGHT_THUMB,
    mp_pose.PoseLandmark.LEFT_HIP,
    mp_pose.PoseLandmark.RIGHT_HIP,
    mp_pose.PoseLandmark.LEFT_KNEE,
    mp_pose.PoseLandmark.RIGHT_KNEE,
    mp_pose.PoseLandmark.LEFT_ANKLE,
    mp_pose.PoseLandmark.RIGHT_ANKLE,
    mp_pose.PoseLandmark.LEFT_HEEL,
    mp_pose.PoseLandmark.RIGHT_HEEL,
    mp_pose.PoseLandmark.LEFT_FOOT_INDEX,
    mp_pose.PoseLandmark.RIGHT_FOOT_INDEX
]

# Load the YOLO model for person detection
yolo_model = YOLO("yolo11n-pose.pt")  # Load your YOLO model

def resize_frame(frame, width=224, height=224):
    """Resize the frame to the specified width and height."""
    return cv2.resize(frame, (width, height))

def sample_and_process_video(video_path, n_frames):
    """Sample n_frames from the video and generate pose estimation data."""
    video_name, _ = os.path.splitext(video_path)
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"Error opening video file: {video_path}")
        return

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Calculate the frame indices to sample
    frame_indices = np.linspace(0, frame_count - 1, n_frames, dtype=int)

    all_features = []

    for frame_id in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
        ret, frame = cap.read()
        if not ret:
            print(f"Failed to read frame {frame_id} from video: {video_path}")
            continue

        # Resize the frame to 224x224
        frame = resize_frame(frame)

        # Detect persons in the frame using YOLO
        results = yolo_model(frame)

        # Store the coordinates of detected persons and their landmarks
        detected_persons = []

        # Loop through each detection
        for detection in results[0].boxes:
            class_id = int(detection.cls[0])  # Get the class ID
            if class_id == 0:  # Check if the detected object is a person
                # Get bounding box coordinates
                xyxy = detection.xyxy[0].tolist()  # Convert to list
                x1, y1, x2, y2 = map(int, xyxy)  # Ensure we convert to integers

                # Crop the detected person from the frame
                person_frame = frame[y1:y2, x1:x2]

                # Convert the cropped frame to RGB for MediaPipe
                rgb_person_frame = cv2.cvtColor(person_frame, cv2.COLOR_BGR2RGB)

                try:
                    # Detect pose landmarks using MediaPipe
                    pose_results = pose.process(rgb_person_frame)

                    if pose_results.pose_landmarks:  # Check if any landmarks were detected
                        keypoints_data = []
                        
                        # Collect keypoint data
                        for landmark in LANDMARKS:
                            landmark_data = pose_results.pose_landmarks.landmark[landmark]
                            keypoints_data.append((landmark_data.x, landmark_data.y, landmark_data.z, landmark_data.visibility))

                        detected_persons.append((x1, y1, x2, y2, keypoints_data))
                except Exception as e:
                    print(f"Error processing frame {frame_id}: {e}")

        # Check if we detected persons and extract their keypoints
        if detected_persons:
            for i, (x1, y1, x2, y2, keypoints) in enumerate(detected_persons):
                row_data = [frame_id, i]  # Frame ID and Person ID (0, 1)
                
                # Assuming keypoints are in the format: [x, y, z, visibility]
                for kp in keypoints:
                    row_data.extend(kp)  # Add x, y, z, visibility to row data

                all_features.append(row_data)
        else:
            print(f"No detected persons in frame {frame_id}.")

    cap.release()
    pose.close()  # Close the MediaPipe Pose object

    # Create DataFrame if we have features collected
    if all_features:
        num_keypoints = len(LANDMARKS)  # Number of landmarks defined in LANDMARKS

        # Create column names using the landmark names
        columns = ['frame_id', 'person'] + \
                  [f'{landmark.name}_x' for landmark in LANDMARKS] + \
                  [f'{landmark.name}_y' for landmark in LANDMARKS] + \
                  [f'{landmark.name}_z' for landmark in LANDMARKS] + \
                  [f'{landmark.name}_visibility' for landmark in LANDMARKS]
        
        df = pd.DataFrame(all_features, columns=columns)

        # Save DataFrame to CSV if not empty
        df.to_csv(video_name + '_mediapipe_pose_landmarks_sampled.csv', index=False)
        print("Pose landmarks with visibility saved to", video_name + '_mediapipe_pose_landmarks_sampled.csv')
    else:
        print("No data collected. The DataFrame is empty.")

# Example usage
video_path = '/Users/jeremiah/video-p-reaction-joketype-modular/experiments/media_pipe_pose_try_october_17.mp4'
n_frames = 30  # Specify the number of frames to sample
sample_and_process_video(vid_1, n_frames)


0: 640x640 3 persons, 85.3ms
Speed: 1.6ms preprocess, 85.3ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)


I0000 00:00:1730238491.301727 14927701 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3 Pro
W0000 00:00:1730238491.376968 14966816 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1730238491.394620 14966826 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.



0: 640x640 3 persons, 62.4ms
Speed: 1.0ms preprocess, 62.4ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 56.8ms
Speed: 1.1ms preprocess, 56.8ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)


/opt/anaconda3/envs/videop/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '



0: 640x640 2 persons, 64.7ms
Speed: 1.4ms preprocess, 64.7ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 62.9ms
Speed: 1.3ms preprocess, 62.9ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 74.6ms
Speed: 1.3ms preprocess, 74.6ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 59.6ms
Speed: 1.5ms preprocess, 59.6ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 3 persons, 58.8ms
Speed: 1.3ms preprocess, 58.8ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 60.5ms
Speed: 1.3ms preprocess, 60.5ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 59.8ms
Speed: 1.5ms preprocess, 59.8ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 3 persons, 55.0ms
Speed: 1.3ms preprocess, 55.0ms inference, 0.3ms postprocess per image at shape (

In [21]:
vid_3 = "/Users/jeremiah/video-p-reaction-joketype-modular/fifteen_twenty_five/"
import os

video_list = os.listdir(vid_3)

for vid in video_list:
    vid_path = vid_3 + vid
    sample_and_process_video(vid_path,n_frames=10)


0: 640x640 2 persons, 87.7ms
Speed: 1.8ms preprocess, 87.7ms inference, 2.9ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 61.0ms
Speed: 1.0ms preprocess, 61.0ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 60.7ms
Speed: 1.8ms preprocess, 60.7ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)


/opt/anaconda3/envs/videop/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '



0: 640x640 3 persons, 63.7ms
Speed: 1.2ms preprocess, 63.7ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 63.1ms
Speed: 1.3ms preprocess, 63.1ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 64.6ms
Speed: 1.2ms preprocess, 64.6ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 69.7ms
Speed: 1.3ms preprocess, 69.7ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 63.0ms
Speed: 1.2ms preprocess, 63.0ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 66.8ms
Speed: 1.2ms preprocess, 66.8ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 59.7ms
Speed: 1.7ms preprocess, 59.7ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)
Pose landmarks with visibility saved to /Users/jeremiah/video-p-reaction-joketype-modular/fifteen_twenty_five/L

ValueError: _graph is None in SolutionBase

## Trying

In [80]:
folder_path = "/Users/jeremiah/video-p-reaction-joketype-modular/fifteen_twenty_five/"

In [81]:
import cv2
import numpy as np
import pandas as pd
import os

In [85]:
# Function to process individual videos and combine data
def process_videos(video_paths):
    combined_features = []
    max_frames = 0  # Keep track of the maximum frame count

    for video_path in video_paths:
        features = process_video2(video_path)  # Assuming process_video returns a DataFrame
        max_frames = max(max_frames, len(features))
        combined_features.append(features)

    # Normalize sizes and pad/truncate frames
    for i in range(len(combined_features)):
        current_length = len(combined_features[i])
        if current_length < max_frames:
            padding = pd.DataFrame(np.zeros((max_frames - current_length, combined_features[i].shape[1])),
                                    columns=combined_features[i].columns)
            combined_features[i] = pd.concat([combined_features[i], padding], ignore_index=True)
        elif current_length > max_frames:
            combined_features[i] = combined_features[i].iloc[:max_frames]

    # Combine all features into one DataFrame
    final_combined_df = pd.concat(combined_features, ignore_index=True)
    return final_combined_df


In [90]:
# Function to process all videos in a folder
def process_videos_in_folder(folder_path):
    combined_features = []
    max_frames = 0  # Keep track of the maximum frame count

    # Loop through each file in the specified folder
    for filename in os.listdir(folder_path):
        video_path = os.path.join(folder_path, filename)
        
        # Check if the file is a video (you can add more formats as needed)
        if filename.endswith(('.mp4', '.avi', '.mov', '.mkv')):
            try:
                features = process_video2(video_path)  # Process the video to get features
                max_frames = max(max_frames, len(features))
                combined_features.append(features)
            except Exception as e:
                print(f"Error processing {video_path}: {e}")

    # Normalize sizes and pad/truncate frames
    for i in range(len(combined_features)):
        current_length = len(combined_features[i])
        if current_length < max_frames:
            padding = pd.DataFrame(np.zeros((max_frames - current_length, combined_features[i].shape[1])),
                                    columns=combined_features[i].columns)
            combined_features[i] = pd.concat([combined_features[i], padding], ignore_index=True)
        elif current_length > max_frames:
            combined_features[i] = combined_features[i].iloc[:max_frames]

    # Combine all features into one DataFrame
    final_combined_df = pd.concat(combined_features, ignore_index=True)
    return final_combined_df

In [91]:
# Example usage

combined_data = process_videos_in_folder(folder_path)


0: 480x640 2 persons, 118.8ms
Speed: 2.9ms preprocess, 118.8ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)
Error processing /Users/jeremiah/video-p-reaction-joketype-modular/fifteen_twenty_five/LJLA66.joke1.rep1.take1.ThatsNotACat.mp4: _graph is None in SolutionBase

0: 480x640 1 person, 50.8ms
Speed: 1.0ms preprocess, 50.8ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)
Error processing /Users/jeremiah/video-p-reaction-joketype-modular/fifteen_twenty_five/QS7TWC.joke2.rep1.take1.TearingPaper.mp4: _graph is None in SolutionBase

0: 480x640 2 persons, 34.5ms
Speed: 0.8ms preprocess, 34.5ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)
Error processing /Users/jeremiah/video-p-reaction-joketype-modular/fifteen_twenty_five/X2BTQN.joke1.rep2.take1.ThatsNotACat.mp4: _graph is None in SolutionBase

0: 480x640 3 persons, 33.3ms
Speed: 0.5ms preprocess, 33.3ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)
Error proces

ValueError: No objects to concatenate

: 

# Good POSE ESTIMATION with MEDIAPIPE and YOLO

In [ ]:
import cv2
import pandas as pd
import numpy as np
from ultralytics import YOLO
import mediapipe as mp

# Load the YOLOv8 model
model = YOLO("yolo11n.pt")  # Ensure the path to your model is correct

# Initialize Mediapipe for pose detection
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5)

# Function to extract body landmarks and track individuals
def process_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frame_id = 0
    all_features = []

    # Initialize a dictionary to keep track of detected individuals
    person_id_dict = {}
    person_counter = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Detect persons in the frame using YOLOv8
        results = model(frame)

        # Extract detections for 'person' class (assuming class ID for person is 0)
        for detection in results[0].boxes:
            class_id = int(detection.cls)  # Get the class ID
            if class_id == 0:  # Check if the detected object is a person
                # Get bounding box coordinates
                x1, y1, x2, y2 = map(int, detection.xyxy[0])  # Convert to integers
                conf = detection.conf[0].item()  # Get confidence score

                # Calculate center coordinates for tracking
                center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2

                # Assign or get the unique person ID based on the center point
                person_id = person_id_dict.get((center_x, center_y), person_counter)
                person_id_dict[(center_x, center_y)] = person_id
                if person_id == person_counter:
                    person_counter += 1

                # Draw bounding box and person ID on the frame
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, f'ID: {person_id}', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

                # Extract pose landmarks using Mediapipe
                image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results_pose = pose.process(image_rgb)

                if results_pose.pose_landmarks:
                    # Create row for the current person
                    row_data = [frame_id, person_id]
                    for landmark in mp_pose.PoseLandmark:
                        landmark_idx = landmark.value
                        landmark_data = results_pose.pose_landmarks.landmark[landmark_idx]
                        
                        # Check visibility and extract coordinates and confidence
                        if landmark_data.visibility > 0:  # Check visibility
                            x = landmark_data.x
                            y = landmark_data.y
                            confidence = landmark_data.visibility
                            row_data.extend([x, y, confidence])
                        else:
                            row_data.extend([None, None, None])  # If not visible, add None
                    all_features.append(row_data)

        frame_id += 1

    cap.release()

    # Create DataFrame
    columns = ['frame_id', 'person'] + \
              [f'{mp_pose.PoseLandmark(i).name}_x' for i in range(33)] + \
              [f'{mp_pose.PoseLandmark(i).name}_y' for i in range(33)] + \
              [f'{mp_pose.PoseLandmark(i).name}_confidence' for i in range(33)]
    
    df = pd.DataFrame(all_features, columns=columns)

    # Save DataFrame to CSV
    df.to_csv('yolov8_pose_landmarks_two_people_per_frame_with_confidence.csv', index=False)
    print("Pose landmarks with confidence saved to yolov8_pose_landmarks_two_people_per_frame_with_confidence.csv")

# Good Medipipe

In [14]:
import cv2
import mediapipe as mp
import pandas as pd

# Initialize MediaPipe Pose model
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5, min_tracking_confidence=0.5)

# Define the specific landmarks to track
LANDMARKS = [
    mp_pose.PoseLandmark.NOSE,
    mp_pose.PoseLandmark.LEFT_EYE_INNER,
    mp_pose.PoseLandmark.LEFT_EYE,
    mp_pose.PoseLandmark.LEFT_EYE_OUTER,
    mp_pose.PoseLandmark.RIGHT_EYE_INNER,
    mp_pose.PoseLandmark.RIGHT_EYE,
    mp_pose.PoseLandmark.RIGHT_EYE_OUTER,
    mp_pose.PoseLandmark.LEFT_EAR,
    mp_pose.PoseLandmark.RIGHT_EAR,
    mp_pose.PoseLandmark.MOUTH_LEFT,
    mp_pose.PoseLandmark.MOUTH_RIGHT,
    mp_pose.PoseLandmark.LEFT_SHOULDER,
    mp_pose.PoseLandmark.RIGHT_SHOULDER,
    mp_pose.PoseLandmark.LEFT_ELBOW,
    mp_pose.PoseLandmark.RIGHT_ELBOW,
    mp_pose.PoseLandmark.LEFT_WRIST,
    mp_pose.PoseLandmark.RIGHT_WRIST,
    mp_pose.PoseLandmark.LEFT_PINKY,
    mp_pose.PoseLandmark.RIGHT_PINKY,
    mp_pose.PoseLandmark.LEFT_INDEX,
    mp_pose.PoseLandmark.RIGHT_INDEX,
    mp_pose.PoseLandmark.LEFT_THUMB,
    mp_pose.PoseLandmark.RIGHT_THUMB,
    mp_pose.PoseLandmark.LEFT_HIP,
    mp_pose.PoseLandmark.RIGHT_HIP,
    mp_pose.PoseLandmark.LEFT_KNEE,
    mp_pose.PoseLandmark.RIGHT_KNEE,
    mp_pose.PoseLandmark.LEFT_ANKLE,
    mp_pose.PoseLandmark.RIGHT_ANKLE,
    mp_pose.PoseLandmark.LEFT_HEEL,
    mp_pose.PoseLandmark.RIGHT_HEEL,
    mp_pose.PoseLandmark.LEFT_FOOT_INDEX,
    mp_pose.PoseLandmark.RIGHT_FOOT_INDEX
]

# Function to extract specific pose landmarks with confidence
def extract_landmarks(results):
    landmarks_dict = {landmark: [None, None, None, None] for landmark in LANDMARKS}  # x, y, z, confidence
    if results.pose_landmarks:
        for idx, landmark in enumerate(results.pose_landmarks.landmark):
            if mp_pose.PoseLandmark(idx) in LANDMARKS:
                landmarks_dict[mp_pose.PoseLandmark(idx)] = [
                    landmark.x, landmark.y, landmark.z, landmark.visibility
                ]
    return landmarks_dict

# Initialize video capture
cap = cv2.VideoCapture(vid_1)

all_features = []pose_extration_mediapipe_yolo
frame_id = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Convert the frame to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Detect pose for the first person
    results_1 = pose.process(frame_rgb)

    # Flip the frame horizontally to detect the second person
    frame_flipped = cv2.flip(frame_rgb, 1)
    results_2 = pose.process(frame_flipped)

    # Extract landmarks for both persons
    features_1 = extract_landmarks(results_1) if results_1.pose_landmarks else {landmark: [None, None, None, None] for landmark in LANDMARKS}
    features_2 = extract_landmarks(results_2) if results_2.pose_landmarks else {landmark: [None, None, None, None] for landmark in LANDMARKS}

    # Append person 1 features
    row_1 = [frame_id, 0]  # Person 1, frame id
    for landmark in LANDMARKS:
        row_1.extend(features_1[landmark])  # Add x, y, z, confidence for each landmark
    all_features.append(row_1)

    # Append person 2 features
    row_2 = [frame_id, 1]  # Person 2, frame id
    for landmark in LANDMARKS:
        row_2.extend(features_2[landmark])  # Add x, y, z, confidence for each landmark
    all_features.append(row_2)

    frame_id += 1

# Release the capture and close windows
cap.release()

# Define column names
columns = ['frame_id', 'person']
for landmark in LANDMARKS:
    columns.extend([f'{landmark.name}_x', f'{landmark.name}_y', f'{landmark.name}_z', f'{landmark.name}_confidence'])

# Create DataFrame
df = pd.DataFrame(all_features, columns=columns)

# Save DataFrame to a CSV file
df.to_csv('pose_landmarks_two_people_per_frame_with_confidence.csv', index=False)

print("Pose landmarks with confidence saved to pose_landmarks_two_people_per_frame_with_confidence.csv")


I0000 00:00:1729164081.111268 1456042 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3 Pro
W0000 00:00:1729164081.220888 1541242 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1729164081.230406 1541252 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/opt/anaconda3/envs/videop/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Pose landmarks with confidence saved to pose_landmarks_two_people_per_frame_with_confidence.csv


In [ ]:


def extract_openpose_features(frames):
    # OpenPose parameters
    params = dict()
    params["model_folder"] = "path_to_openpose_models/"  # Update this path
    params["face"] = False
    params["hand"] = False

    # Initialize OpenPose
    op_wrapper = op.WrapperPython()
    op_wrapper.configure(params)
    op_wrapper.start()

    # Prepare lists to hold features and column names
    all_features = []
    column_names = []

    for frame_idx, frame in enumerate(frames):
        datum = op.Datum()
        datum.cvInputData = frame
        op_wrapper.emplaceAndPop([datum])

        # Get pose keypoints
        keypoints = datum.poseKeypoints  # Shape: (num_people, 25, 3)
        
        # Initialize features for the frame
        frame_features = []
        
        if keypoints is not None and len(keypoints.shape) == 3:
            num_people = keypoints.shape[0]
            # If more than two people, select two based on average confidence
            if num_people >= 2:
                # Calculate average confidence for each person
                avg_confidences = keypoints[:, :, 2].mean(axis=1)
                # Get indices of the two people with the highest average confidence
                top_indices = np.argsort(-avg_confidences)[:2]
            else:
                top_indices = [0]
                # If only one person detected, duplicate index to maintain two people
                if num_people == 1:
                    top_indices = [0, 0]
        else:
            # No people detected
            top_indices = []
            frame_features.extend([0] * 75 * 2)  # 75 features per person, two people
            all_features.append(frame_features)
            continue  # Move to the next frame

        # Extract features for the two selected people
        for person_idx in range(2):
            if person_idx < len(top_indices):
                idx = top_indices[person_idx]
                person_keypoints = keypoints[idx]
                flattened = person_keypoints.flatten().tolist()
            else:
                # Fill with zeros if fewer than two people
                flattened = [0] * 75
            frame_features.extend(flattened)

        all_features.append(frame_features)

        # Generate column names (only once)
        if not column_names:
            for person_num in range(1, 3):
                for kp_idx in range(25):
                    for coord in ['x', 'y', 'c']:
                        col_name = f'person{person_num}_kp{kp_idx}_{coord}'
                        column_names.append(col_name)

    # Create DataFrame
    df = pd.DataFrame(all_features, columns=column_names)
    return df

def calculate_interaction_features(df):
    interaction_features = []
    for idx, row in df.iterrows():
        person1_x = row[[f'person1_kp{kp_idx}_x' for kp_idx in range(25)]].values
        person1_y = row[[f'person1_kp{kp_idx}_y' for kp_idx in range(25)]].values
        person2_x = row[[f'person2_kp{kp_idx}_x' for kp_idx in range(25)]].values
        person2_y = row[[f'person2_kp{kp_idx}_y' for kp_idx in range(25)]].values

        # Example interaction feature: distance between noses (keypoint 0)
        if person1_x[0] != 0 and person2_x[0] != 0:
            distance = np.sqrt((person1_x[0] - person2_x[0])**2 + (person1_y[0] - person2_y[0])**2)
        else:
            distance = np.nan  # Missing data

        interaction_features.append(distance)

    df['nose_distance'] = interaction_features
    return df

def main():
    # Read video frames
    cap = cv2.VideoCapture('video.mp4')  # Update with your video file path
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()

    # Extract features using OpenPose
    features_df = extract_openpose_features(frames)

    # Calculate interaction features
    features_df = calculate_interaction_features(features_df)

    # Save to CSV or inspect
    features_df.to_csv('openpose_features.csv', index=False)
    print(features_df.head())

if __name__ == "__main__":
    main()
